## tl;dr
Recompute two archived LFQ comparisons from six report extracts. This does not run SynapSpec, DIA-NN or Spectronaut. Outputs are provisional comparisons, not current-release rankings.

## Context & Methods
Precursor q-value < 0.01; decoys excluded. Keys: exact modified sequence plus charge. CV is computed within A or B, using all three positive finite replicate quantities. LFQ ratios reuse the DeepMSFlow BION preset and precursor MS2 aggregation.

### Key Assumptions
Input stems must match across all three tools. Folder version labels are not independently verified releases. Search settings, FASTA, MBR configuration, runtime and hardware are not fully reconciled. Separate MBR reports are excluded. Internal LFQBench is not asserted to be PXD028735.

## Data
Set `BENCHMARK_INPUTS`, `BENCHMARK_SCRIPTS`, and `DEEPMSFLOW_ROOT` before launching the kernel. Source manifest and extract hashes are written to a private provenance file, never the public website.

In [ ]:
import os
import sys
import tempfile
from pathlib import Path

input_directory = Path(os.environ['BENCHMARK_INPUTS'])
script_directory = Path(os.environ['BENCHMARK_SCRIPTS'])
parser_directory = Path(os.environ['DEEPMSFLOW_ROOT']) / 'instrumentation'
sys.path[:0] = [str(script_directory), str(parser_directory)]
from audit_inputs import audit_report
from export_comparison import export

audits = [audit_report(path) for path in sorted(input_directory.glob('*.parquet'))]
assert len(audits) == 6
assert all(item['status'] == 'input_checks_passed' for item in audits)
[(item['file'], item['precursor_runs'], item['quantity_conflicts']) for item in audits]

## Results
Each execution writes a new output directory, leaving all source reports and previous exports untouched.

In [ ]:
output_directory = Path(tempfile.mkdtemp(prefix='lfq-notebook-')) / 'output'
records = export(input_directory, output_directory)
from IPython.display import Image, display
for record in records:
    display(Image(filename=str(output_directory / record['figure'])))
print('Outputs:', output_directory)

## Takeaways
Use the generated tables and images together. ID count measures coverage, completeness uses each tool's own detected population, and CV uses only complete three-replicate observations. Different eligible populations and unverified full settings prevent a definitive tool ranking. Protein grouping is tool-specific and not used as a cross-tool score. Review sample preparation records before publishing expected-ratio claims as a validated benchmark.